# rangelab — finding ranges worth targeting

Built around your goal: ranges of **≥10 minutes** that either break in a
predictable direction, or break and then revert. The pipeline is

```
bars -> ranges (+features) -> forward outcomes -> scored cells -> shortlist
```

It reuses the conventions in your own notebooks (18:00 ET anchor,
`op_bucket` edges, `SHRINK_K = 150`, the 2019-06-01 train/test split) so results
line up with what you already have. What it adds is the **selection layer**:
a within-day permutation null that tells you how good the best cell in the grid
looks when there is no edge at all.

Read `FINDINGS.md` first — the headline results change how you should read
everything below.

## 0. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve()))   # rangelab/ sits next to this notebook

import numpy as np, pandas as pd
from rangelab.pipeline import run, build, diagnostics_table, describe_cell, FAMILIES
from rangelab.ranges import build_ranges
from rangelab.outcomes import forward_outcomes
from rangelab.cells import score_cells, shortlist

pd.set_option("display.width", 260); pd.set_option("display.max_columns", 60)
NQ = "nq_15y_1min.csv"
for k, v in FAMILIES.items():
    print(f"{k:16s} {v}")

## 1. Ranges and their features

A range is a fixed-duration clock window anchored at 18:00 ET. Features split
into three groups, all knowable the moment the range closes:

**Position** — `op` / `op_bucket`: where this range opened inside the previous
one. This is the conditioner your MTF notebook is built on, with identical edges.

**Character** — `width_pct_rank` and `width_z` (width against its own trailing
250 ranges), `close_pos` (where it closed inside itself), `formed_first` (which
extreme came first), `overlap_rel` / `overlap_frac` (inside / engulfs / gapped
vs the previous range), `vol_balance` (volume above vs below its own mid), and
`rel_volume`. The volume ones use the `Volume` column none of your notebooks
touch.

**Regime** — `atr_pct_rank` and `atr_zone`, Wilder-smoothed on the range series.

Two filters matter and both exist because of bugs I hit building this:

* `ok_coverage` — a range must hold ≥60% of its expected 1-min bars.
* `ok_forward` — its forward horizon must be ≥90% complete. Slots near 16:00
  have their horizon cut short by the 17:00 close, and without this filter
  "did not break at 16:45" looks like a signal when it is an absence of data.

`usable` = both, plus an adjacent previous range.

In [ ]:
from rangelab.data import load_bars
bars = load_bars(NQ, symbol="NQ", start="2013-01-01", end="2025-03-14")
r60 = build_ranges(bars, 60)
o60 = forward_outcomes(bars, r60, horizon_mult=3.0)
print(f"{len(o60):,} 60-min ranges | usable {o60['usable'].mean():.1%}")
o60.loc[o60['usable'], ['slot','width','width_pct_rank','close_pos','formed_first',
                        'op_bucket','overlap_rel','vol_balance','atr_zone',
                        'break_side','cont_R','rev_mid','rev_far_edge']].head(8)

## 2. The forward outcome tree

Per range, over a horizon of `horizon_mult` × the range duration (default 3, the
same multiple `magic_hours` uses):

* `break_side` — which edge went first (`High` / `Low` / `none`), `mins_to_break`
* `cont_R` — how far past the broken edge, in range-widths
* `rev_mid`, `rev_range_open`, `rev_far_edge` — did it come back to each, and
  `mins_to_*` for each
* `mae_to_mid_R` — adverse excursion with walk-away truncation at the mid, as in
  `magic_hours`
* `cont_1p0R`, `cont_not_rev` — magnitude-conditioned, so a direction call is
  judged alongside whether it actually travelled

In [ ]:
u = o60[o60['usable']]
br = u[u['broke']]
print(f"broke within 3h: {u['broke'].mean():.1%}   (High {100*(br['break_side']=='High').mean():.1f}%)")
for t in ("mid","range_open","far_edge"):
    print(f"  back to {t:11s}: {100*br[f'rev_{t}'].mean():5.1f}%   median {br[f'mins_to_{t}'].median():>4.0f} min")
print(f"\ncontinuation past the edge: median {br['cont_R'].median():.2f}R, p90 {br['cont_R'].quantile(.9):.2f}R")

## 3. Scoring cells — and the selection test

A cell is one combination of conditioners. Scoring is the easy part; knowing
which to believe is not. Four defences, in increasing order of what they tell you:

1. **Shrinkage** toward the pooled bucket curve, weight `n/(n+150)`, fitted on
   train only — your `SHRINK_K` approach, unchanged.
2. **Day-block bootstrap** CIs. Worth knowing: this turned out to barely matter
   here. When a cell is keyed on `slot`, each trading day contributes exactly one
   observation, so there is no within-day clustering to correct. Measured
   inflation vs Wilson is ~0.99×. It still protects you for pooled cells.
3. **Within-day permutation null** — shuffle the outcome among ranges of the same
   day, recompute the whole grid, repeat. Gives a per-cell p-value that respects
   serial correlation *and* the distribution of the best cell under no edge.
4. **Benjamini-Hochberg FDR** across cells.

The number to look at is `best_vs_null`: the observed best edge divided by the
95th percentile of the best-cell edge under the null. **Below 1 means the best
cell in your grid is no better than luck.**

In [ ]:
res = run(NQ, symbol="NQ", durations=(15,30,60,120,240),
          by=("duration","slot","op_bucket"), min_n=50,
          n_boot=200, n_perm=200, split_date="2019-06-01", seed=1)
diagnostics_table(res["diag"])

`break_up` comes out **below 1**. The best directional-breakout cell in a
1,737-cell grid is no better than the best cell you'd get by shuffling the
labels. On this data, with this conditioning, direction is not predictable.

The reversion families sit at 1.8–2.3×, which is real — but section 4 shows it
collapses to a single variable.

## 4. The width effect — why most of the grid was a proxy

The reversion survivors cluster at 15-min 09:00, 240-min 22:00, 60-min 07:00 —
all low-volatility pre-open and Asia windows — and spread evenly across 9 or 10
of the 12 `op_bucket`s. When a conditioner's survivors spread evenly across
another conditioner, the second one is contributing nothing.

What they share is tight ranges. And because `cont_R` and the reversion targets
are both measured in **range-widths**, a tight range trivially produces large
moves in *both* directions:

In [ ]:
u = res["ranges"]
u = u[u["usable"]].assign(w_atr=lambda x: x["width"]/x["atr"])
br = u[u["broke"]].copy()
br["w_atr_q"] = pd.qcut(br["w_atr"], 5, labels=["1 tightest","2","3","4","5 widest"])
g = (br.groupby("w_atr_q", observed=True)
       .agg(n=("rev_mid","size"), rev_mid=("rev_mid","mean"),
            rev_far=("rev_far_edge","mean"), cont_1R=("cont_1p0R","mean"),
            cont_R_med=("cont_R","median")))
g[["rev_mid","rev_far","cont_1R"]] = (100*g[["rev_mid","rev_far","cont_1R"]]).round(1)
g.round(2)

Reversion **and** continuation both rise as the range tightens. That is the
signature of a denominator artifact: a 1R move is a small move when the range is
small. The slot cells were proxies for "this window usually has tight ranges".

The practical consequence: **`width ÷ ATR` is the variable to select ranges on**,
and it is knowable before the range breaks. But measure your targets in ATR units,
not range-widths, or you are only rediscovering that small ranges are small.

In [ ]:
# Does slot still add anything once width is controlled? Compare best_vs_null.
o = res["ranges"].assign(w_atr=lambda x: x["width"]/x["atr"])
o["w_atr_b"] = o.groupby("duration")["w_atr"].transform(
    lambda s: pd.qcut(s, 5, labels=["1 tight","2","3","4","5 wide"]))

rows=[]
for label, by in {"slot x op_bucket": ["duration","slot","op_bucket"],
                  "width only":       ["duration","w_atr_b"],
                  "width x slot":     ["duration","w_atr_b","slot"]}.items():
    for fam in ("rev_mid","rev_far_edge"):
        c,d = score_cells(o, fam, by=by, min_n=50, n_boot=100, n_perm=100, seed=3)
        rows.append({"conditioners":label, "family":fam, "cells":d["cells_tested"],
                     "best_edge_pp":round(100*d["observed_best_edge"],1),
                     "null_p95_pp":round(100*d["null_best_edge_p95"],1),
                     "best_vs_null":round(d["observed_best_edge"]/d["null_best_edge_p95"],2)})
pd.DataFrame(rows)

## 5. Inspecting a candidate

For any shortlisted cell, `describe_cell` gives the probabilities *and* the
magnitudes together — because a direction call without follow-through is not a
trade idea.

In [ ]:
sl = res["shortlist"]
print(f"{len(sl)} cells cleared FDR + out-of-sample + year-agreement + bootstrap")
print(sl["family"].value_counts().to_string())
sl.head(10)[["family","duration","slot","op_bucket","n","rate","base",
             "train_shrunk","test_rate","boot_lo","boot_hi","q_value","year_agree_frac"]]

In [ ]:
if len(sl):
    row = sl.iloc[0]
    display(describe_cell(res["ranges"], row))

## 6. Where to take it

The pipeline is deliberately strict, and the strictness is the point — it told
us that the directional half of the question has no answer in this grid, which
saves you from trading it.

Things worth trying next, roughly in order of expected payoff:

1. **Re-express targets in ATR units** rather than range-widths, then re-run the
   reversion families. If an edge survives *that*, it is not the width artifact.
2. **Condition on `width ÷ ATR` first**, then look for anything that adds to it.
   The cell grid shrinks enormously, which lowers the null bar and raises power.
3. **Use the unused features.** `vol_balance`, `rel_volume`, `close_pos`,
   `formed_first` and `overlap_rel` are all computed and none is in your current
   notebooks. `close_pos` in particular is a plausible directional conditioner
   that `op_bucket` cannot capture.
4. **Cross-symbol replication on GC.** You chose NQ-only for now, but a cell that
   holds on both is far stronger evidence. The loader already handles GC's
   different timezone.
5. **Widen the horizon sweep.** Everything here uses `horizon_mult=3`. Whether an
   edge is horizon-specific is diagnostic: a real structural effect should degrade
   smoothly, not appear only at one multiple.

One warning the pipeline cannot enforce. Every time you add a conditioner you
multiply the cells, which raises the null bar roughly as the square root of the
count while thinning each cell. There is an optimal granularity and it is coarser
than feels satisfying. The `best_vs_null` column is how you find it — run the same
family at several granularities and keep the one with the highest ratio, not the
highest raw rate.